# Extract Scalar Performance of models from a log file

## Fichiers

+ E13sg_8-9_run_all_agents_test.log, expe version minor 8 et 9 AVEC (!) la latence dans les observations, Randtype Gaussian
+ E13sg_10_run_all_agents_test.log, expe SANS la latence dans les observations, Randtype Gaussian 
+ E13sgu_11-12_a2c_run_all_agents_test.log, expe SANS la latence dans les observations, Randtypes Gaussian+Uniform 

In [29]:
# Chemin vers votre fichier de log
log_file_path = 'results/E13sg_8-9_run_all_agents_test.log'

In [30]:
import os
import re
import pandas as pd

# Listes pour stocker les données extraites
data = []

# Expressions régulières
pattern_result = re.compile(
    r'Trial:\s*(\d+).*?TestPerf\(ScalarPerf,USLA,RAM,SLAV\):\s*\(([^,]+),\s*([^,]+),\s*([^,]+),\s*([^\)]+)\)'
)

pattern_agent = re.compile(
    r'AgentFilename:\s*\.\/outputs\/([^\/]+)\/agent-([^_]+)-([^_]+)_'
)

# Variable pour stocker temporairement les infos Agent
current_agent_info = None

# Fonction pour extraire "version" et "simu" à partir du nom de dossier
def extract_version_simu(folder_name):
    # Exemple : E13sg_9_tabddpm-a2c-training_disq
    parts = folder_name.split('_')
    if len(parts) >= 3:
        version = parts[0]
        version_minor = parts[1]  # E13sg_9
        simu = parts[2].split('-')[0]     # tabddpm
        return version, version_minor, simu
    return None, None, None

# Lecture du fichier de log
with open(log_file_path, 'r') as file:
    for line in file:
        # Vérifier si la ligne est une ligne AgentFilename
        match_agent = pattern_agent.search(line)
        if match_agent:
            folder_name = match_agent.group(1)
            version, version_minor, simu = extract_version_simu(folder_name)
            model = match_agent.group(3)
            # Stocker temporairement ces infos
            current_agent_info = {
                'Version': version,
                'VersionMinor': version_minor,
                'Simu': simu,
                'Model': model
            }
        # Vérifier si la ligne est une ligne TestPerf
        match_result = pattern_result.search(line)
        if match_result:
            trial_num = int(match_result.group(1))
            scalar_perf = float(match_result.group(2))
            usla = int(match_result.group(3))
            ram = int(match_result.group(4))
            slav = int(match_result.group(5))
            # Créer l'entrée avec les résultats
            entry = {
                'Trial': trial_num,
                'ScalarPerf': scalar_perf,
                'USLA': usla,
                'RAM': ram,
                'SLAV': slav,
                'Version': None,
                'VersionMinor': None,
                'Simu': None,
                'Model': None
            }
            # Ajouter les infos Agent si disponibles
            if current_agent_info:
                entry.update(current_agent_info)
            data.append(entry)

# Convertir en DataFrame
dffull = pd.DataFrame(data)

base_name, _ = os.path.splitext(log_file_path)
expcsv=base_name+".csv"
print(f"Save results dataset to: {expcsv}...")
dffull.to_csv(expcsv, index=False)

# Afficher le DataFrame final
#print(df)
version_list = dffull['Version'].unique()
version_minor_list = dffull['VersionMinor'].unique()
simu_list = dffull['Simu'].unique()
model_list = dffull['Model'].unique()
print("Versions found in the log file:")
print(version_list)
print(version_minor_list)
print(simu_list)
print(model_list)
roundat=3

pd.options.display.width = 500

# Generate result table
for version in version_list:
    results = []
    for version_minor in version_minor_list:
        for simu in simu_list:
            for model in model_list:
                dfperf = dffull.loc[(dffull['Version'] == version) & (dffull['VersionMinor'] == version_minor) & (dffull['Simu'] == simu) & (dffull['Model'] == model)]
                if not dfperf.empty:
                    #dfperf = dffull.loc[(dffull['Version'] == version) & (dffull['VersionMinor'] == version_minor) & (dffull['Simu'] == simu) & (dffull['Model'] == model)]
                    scalar_perf_min = dfperf['ScalarPerf'].min()
                    scalar_perf_max = dfperf['ScalarPerf'].max()
                    dfmin = dfperf.loc[dfperf['ScalarPerf'] == scalar_perf_min]
                    dfmax = dfperf.loc[dfperf['ScalarPerf'] == scalar_perf_max]

                    scalar_perf_max = round(dfperf['ScalarPerf'].max(), roundat)
                    scalar_perf_min = round(scalar_perf_min, roundat)
                    scalar_perf_mean = round(dfperf['ScalarPerf'].mean(), roundat)
                    scalar_perf_std = round(dfperf['ScalarPerf'].std(), roundat)
                    entry = {
                        'Version': version+"_"+version_minor,
                        'Dataset': str(simu).upper(),
                        'Model': model,
                        'Perfmin': scalar_perf_min,
                        'Perfmax': scalar_perf_max,
                        'Perfmean': scalar_perf_mean,
                        'Perfstd': scalar_perf_std,
                        'USLAmin': dfmin.iloc[0]["USLA"],
                        'USLAmax': dfmax.iloc[0]["USLA"],
                        'RAMmin': dfmin.iloc[0]["RAM"],
                        'RAMmax': dfmax.iloc[0]["RAM"],
                        'SLAVmin': dfmin.iloc[0]["SLAV"],
                        'SLAVmax': dfmax.iloc[0]["SLAV"],
                    }
                    results.append(entry)

    # Convertir en DataFrame
    dfresults = pd.DataFrame(results)

    # Afficher le DataFrame final
    print(dfresults)


Save results dataset to: results/E13sg_8-9_run_all_agents_test.csv...
Versions found in the log file:
['E13sg']
['8' '9']
['orig' 'dbsas' 'cvae' 'tabddpm']
['SB3A2C' 'SB3PPO' 'SB3DQN']
    Version  Dataset   Model  Perfmin  Perfmax  Perfmean  Perfstd  USLAmin  USLAmax    RAMmin    RAMmax  SLAVmin  SLAVmax
0   E13sg_8     ORIG  SB3A2C    0.343    0.374     0.359    0.010      335      335  28518656  29172096        0        0
1   E13sg_8     ORIG  SB3PPO    0.287    0.323     0.302    0.011      335      335  27329920  28098560        0        0
2   E13sg_8     ORIG  SB3DQN    0.244    1.284     0.590    0.381      352      701  25340160  25309696       13      356
3   E13sg_8    DBSAS  SB3A2C    0.393    0.462     0.409    0.020      336      356  29524736  29726720        1       16
4   E13sg_8    DBSAS  SB3PPO    0.310    0.352     0.322    0.012      335      335  27814912  28708096        0        0
5   E13sg_8    DBSAS  SB3DQN    0.867    6.213     3.359    1.934      553     2391